# Task 050: svmbir-master — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: CT reconstruction using SVMBIR (model-based iterative reconstruction)

**Paper**: Software library with multiple algorithm papers
**Repository**: https://github.com/cabouman/svmbir

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 23.89 dB
- **SSIM**: 0.7975


### Mathematical Formulation

The Radon transform maps the 2D attenuation $f(x,y)$ to sinogram $p(\theta, s)$:

$$p(\theta, s) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

**Filtered Back-Projection (FBP)**:
$$f(x,y) = \int_0^{\pi} \left[ p(\theta, \cdot) \ast g \right](x\cos\theta + y\sin\theta)\,d\theta$$

where $g$ is the ramp filter: $\hat{g}(\omega) = |\omega|$.

**Iterative reconstruction** (SIRT/CGLS):
$$x^{(k+1)} = x^{(k)} + \alpha \, A^T(b - Ax^{(k)})$$

**TV-regularized**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - b\|_2^2 + \lambda \|\nabla x\|_1$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import numpy as np
import scipy.ndimage
import sys
import time
import matplotlib.pyplot as plt

# Try importing optional dependencies for metrics/transforms

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`_gen_ellipse`, `_qggmrf_derivative`, `_compute_prior_gradient`, `load_and_preprocess_data`

In [ ]:
def _gen_ellipse(x_grid, y_grid, x0, y0, a, b, gray_level, theta=0):
    """Generates a single ellipse mask scaled by gray_level."""
    c = np.cos(theta)
    s = np.sin(theta)
    x_rot = (x_grid - x0) * c + (y_grid - y0) * s
    y_rot = -(x_grid - x0) * s + (y_grid - y0) * c
    mask = (x_rot ** 2 / a ** 2 + y_rot ** 2 / b ** 2) <= 1.0
    return mask * gray_level

def _qggmrf_derivative(delta, sigma_x, p, q, T):
    """
    Computes the derivative of the Q-GGMRF potential function w.r.t delta.
    Approximation logic preserved from input code.
    """
    abs_d = np.abs(delta) + 1e-6 
    sign_d = np.sign(delta)
    u = abs_d / T
    
    # rho(x) = |x|^p / (1 + |x/T|^(p-q))
    # Using the derivation logic provided in the input context:
    num = abs_d ** p
    den = 1 + u ** (p - q)
    
    d_num = p * abs_d ** (p - 1) * sign_d
    d_den = (p - q) * (u ** (p - q - 1)) * (1.0/T) * sign_d
    
    grad = (d_num * den - num * d_den) / (den ** 2)
    return grad / (sigma_x ** p)

def _compute_prior_gradient(image, sigma_x, p, q, T):
    """Computes the gradient of the Q-GGMRF prior (Markov Random Field)."""
    total_grad = np.zeros_like(image)
    
    # Right neighbor interaction: x[r,c] - x[r,c+1]
    d = image - np.roll(image, -1, axis=1)
    d[:, -1] = 0 # Boundary
    total_grad += _qggmrf_derivative(d, sigma_x, p, q, T)
    
    # Left neighbor interaction: x[r,c] - x[r,c-1]
    d = image - np.roll(image, 1, axis=1)
    d[:, 0] = 0
    total_grad += _qggmrf_derivative(d, sigma_x, p, q, T)
    
    # Down neighbor interaction: x[r,c] - x[r+1,c]
    d = image - np.roll(image, -1, axis=0)
    d[-1, :] = 0
    total_grad += _qggmrf_derivative(d, sigma_x, p, q, T)
    
    # Up neighbor interaction: x[r,c] - x[r-1,c]
    d = image - np.roll(image, 1, axis=0)
    d[0, :] = 0
    total_grad += _qggmrf_derivative(d, sigma_x, p, q, T)
    
    return total_grad

def load_and_preprocess_data(image_size, num_views, noise_level=0.01):
    """
    Generates synthetic Shepp-Logan phantom data and creates a noisy sinogram.
    
    Returns:
        gt_image (np.array): Ground truth image.
        sinogram (np.array): Observed noisy sinogram.
        angles (np.array): Projection angles in radians.
    """
    # 1. Define geometry
    angles = np.linspace(0, np.pi, num_views, endpoint=False)
    
    # 2. Generate Ground Truth
    gt_image = _gen_shepp_logan(image_size, image_size)
    
    # 3. Simulate "Clean" Sinogram using Forward Operator
    # Note: We call forward_operator locally here to simulate data creation.
    clean_sino = forward_operator(gt_image, angles)
    
    # 4. Add Noise
    noise = np.random.normal(0, noise_level * np.max(clean_sino), clean_sino.shape)
    sinogram = clean_sino + noise
    
    return gt_image, sinogram, angles

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `_gen_shepp_logan`, `_back_project_internal`, `forward_operator`

In [ ]:
def _gen_shepp_logan(num_rows, num_cols):
    """Generates the Shepp-Logan phantom."""
    sl_paras = [
        {'x0': 0.0, 'y0': 0.0, 'a': 0.69, 'b': 0.92, 'theta': 0, 'gray_level': 2.0},
        {'x0': 0.0, 'y0': -0.0184, 'a': 0.6624, 'b': 0.874, 'theta': 0, 'gray_level': -0.98},
        {'x0': 0.22, 'y0': 0.0, 'a': 0.11, 'b': 0.31, 'theta': -18, 'gray_level': -0.02},
        {'x0': -0.22, 'y0': 0.0, 'a': 0.16, 'b': 0.41, 'theta': 18, 'gray_level': -0.02},
        {'x0': 0.0, 'y0': 0.35, 'a': 0.21, 'b': 0.25, 'theta': 0, 'gray_level': 0.01},
        {'x0': 0.0, 'y0': 0.1, 'a': 0.046, 'b': 0.046, 'theta': 0, 'gray_level': 0.01},
        {'x0': 0.0, 'y0': -0.1, 'a': 0.046, 'b': 0.046, 'theta': 0, 'gray_level': 0.01},
        {'x0': -0.08, 'y0': -0.605, 'a': 0.046, 'b': 0.023, 'theta': 0, 'gray_level': 0.01},
        {'x0': 0.0, 'y0': -0.605, 'a': 0.023, 'b': 0.023, 'theta': 0, 'gray_level': 0.01},
        {'x0': 0.06, 'y0': -0.605, 'a': 0.023, 'b': 0.046, 'theta': 0, 'gray_level': 0.01}
    ]
    axis_x = np.linspace(-1.0, 1.0, num_cols)
    axis_y = np.linspace(1.0, -1.0, num_rows)
    x_grid, y_grid = np.meshgrid(axis_x, axis_y)
    image = np.zeros_like(x_grid)
    for el in sl_paras:
        image += _gen_ellipse(x_grid, y_grid, el['x0'], el['y0'], el['a'], el['b'], 
                              el['gray_level'], el['theta'] / 180.0 * np.pi)
    return image

def _back_project_internal(sinogram, angles, image_shape):
    """
    Adjoint of the forward projector.
    """
    if HAS_SKIMAGE:
        # skimage iradon takes (num_det, num_angles)
        sino_T = sinogram.T
        theta_deg = np.degrees(angles)
        # filter_name=None implies unfiltered backprojection (adjoint of Radon)
        recon = iradon(sino_T, theta=theta_deg, output_size=image_shape[0], filter_name=None, circle=True)
        return recon
    else:
        # Manual backprojection
        num_rows, num_cols = image_shape
        backproj = np.zeros(image_shape, dtype=np.float32)
        for i, angle_rad in enumerate(angles):
            angle_deg = np.degrees(angle_rad)
            projection = sinogram[i, :]
            smeared = np.tile(projection, (num_rows, 1))
            rotated_back = scipy.ndimage.rotate(smeared, -angle_deg, reshape=False, order=1, mode='constant', cval=0.0)
            backproj += rotated_back
        return backproj

def forward_operator(x, angles):
    """
    Computes the Radon Transform (Forward Projection).
    
    Args:
        x (np.array): Input image (N, N).
        angles (np.array): Projection angles in radians.
        
    Returns:
        y_pred (np.array): Sinogram (num_angles, num_detectors).
    """
    if HAS_SKIMAGE:
        theta_deg = np.degrees(angles)
        # skimage returns (num_det, num_angles), we want (num_angles, num_det)
        sino = radon(x, theta=theta_deg, circle=True)
        return sino.T
    else:
        num_angles = len(angles)
        num_rows, num_cols = x.shape
        num_det = num_cols 
        sinogram = np.zeros((num_angles, num_det), dtype=np.float32)
        
        for i, angle_rad in enumerate(angles):
            angle_deg = np.degrees(angle_rad)
            rotated_img = scipy.ndimage.rotate(x, angle_deg, reshape=False, order=1, mode='constant', cval=0.0)
            sinogram[i, :] = rotated_img.sum(axis=0)
        return sinogram

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `run_inversion`

In [ ]:
def run_inversion(sinogram, angles, num_iters=100, step_size=0.001, 
                  p=1.2, q=2.0, T=1.0, sigma_x=None):
    """
    Performs MBIR reconstruction using Gradient Descent with Momentum and Q-GGMRF prior.
    
    Args:
        sinogram: Observed data.
        angles: Projection angles.
        num_iters: Number of iterations.
        step_size: Gradient descent step size (alpha).
        p, q, T: Prior parameters.
        sigma_x: Scaling parameter for prior.
    
    Returns:
        result (np.array): Reconstructed image.
    """
    num_angles, num_det = sinogram.shape
    img_shape = (num_det, num_det)
    
    # Heuristic for sigma_x if not provided
    if sigma_x is None:
        val = np.mean(np.abs(sinogram))
        sigma_x = 0.2 * val
    
    # Initialization: Normalized Backprojection
    bp = _back_project_internal(sinogram, angles, img_shape)
    x = bp / num_angles # Initial scaling guess
    
    # Momentum initialization
    velocity = np.zeros_like(x)
    mu = 0.9 # Momentum coefficient
    
    print(f"Starting Inversion: {num_iters} iters, sigma_x={sigma_x:.3f}")
    
    for k in range(num_iters):
        # 1. Forward Project
        Ax = forward_operator(x, angles)
        
        # 2. Residual (Ax - y)
        residual = Ax - sinogram 
        
        # 3. Data Gradient: A.T * (Ax - y)
        grad_data = _back_project_internal(residual, angles, img_shape)
        
        # 4. Prior Gradient
        grad_prior = _compute_prior_gradient(x, sigma_x, p, q, T)
        
        # 5. Total Gradient = Data Term + Prior Term
        total_grad = grad_data + grad_prior
        
        # 6. Update (Momentum + Gradient Descent)
        velocity = mu * velocity - step_size * total_grad
        x = x + velocity
        
        # 7. Positivity Constraint
        x[x < 0] = 0
        
        # Optional: Monitoring
        if k % 20 == 0:
            data_cost = 0.5 * np.sum(residual ** 2)
            print(f"  Iter {k}: Data Cost {data_cost:.2e}")
            
    return x

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(gt, recon, save_path="reconstruction_result.png"):
    """
    Computes PSNR/SSIM and saves a comparison plot.
    
    Returns:
        metrics (dict): Dictionary containing PSNR and SSIM.
    """
    # Normalize for fair metric calculation
    def normalize(arr):
        mn = arr.min()
        mx = arr.max()
        if mx - mn == 0: return arr
        return (arr - mn) / (mx - mn)

    gt_norm = normalize(gt)
    recon_norm = normalize(recon)
    
    # PSNR
    mse = np.mean((gt_norm - recon_norm) ** 2)
    if mse == 0:
        psnr_val = 100.0
    else:
        psnr_val = 20 * np.log10(1.0 / np.sqrt(mse))
        
    # SSIM
    ssim_val = 0.0
    if HAS_SKIMAGE:
        ssim_val = ssim_func(gt_norm, recon_norm, data_range=1.0)
    
    print(f"Evaluation -> PSNR: {psnr_val:.2f} dB, SSIM: {ssim_val:.4f}")
    
    # Visualization
    try:
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        ax[0].imshow(gt, cmap='gray')
        ax[0].set_title("Ground Truth")
        ax[0].axis('off')
        
        ax[1].imshow(recon, cmap='gray')
        ax[1].set_title(f"Reconstruction\nPSNR: {psnr_val:.1f}")
        ax[1].axis('off')
        
        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()
        print(f"Figure saved to {save_path}")
    except Exception as e:
        print(f"Plotting failed: {e}")
        
    return {"psnr": psnr_val, "ssim": ssim_val}

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Configuration
IMG_SIZE = 128
NUM_VIEWS = 180
ITERATIONS = 100
STEP_SIZE = 0.001

### 1. Load Data

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Load Data
print("Step 1: Loading and preprocessing data...")
gt, sino, angles = load_and_preprocess_data(IMG_SIZE, NUM_VIEWS)

### 2. Check Forward Operator (Sanity Check)

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Check Forward Operator (Sanity Check)
print("Step 2: verifying forward operator...")
test_proj = forward_operator(gt, angles)
if test_proj.shape != sino.shape:
    raise ValueError("Forward operator shape mismatch.")

### 3. Run Inversion

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 3. Run Inversion
print("Step 3: Running inversion...")
reconstruction = run_inversion(sino, angles, num_iters=ITERATIONS, step_size=STEP_SIZE)

### 4. Evaluate

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Evaluate
print("Step 4: Evaluating results...")
evaluate_results(gt, reconstruction)

print("OPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **svmbir-master**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=23.89 dB, SSIM=0.7975)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Software library with multiple algorithm papers
- Repository: https://github.com/cabouman/svmbir
